In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class KVCache:
    def __init__(self):
        self.cache = {"key": None, "value": None}

    def update(self, key, value):
        # key, value shapes: [batch, heads, seq_len, head_dim]
        if self.cache["key"] is None:
            self.cache["key"] = key
            self.cache["value"] = value
        else:
            self.cache["key"] = torch.cat([self.cache["key"], key], dim=2)
            self.cache["value"] = torch.cat([self.cache["value"], value], dim=2)

    def get_cache(self):
        return self.cache

    def reset(self):
        self.cache = {"key": None, "value": None}

class SelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.query_proj = nn.Linear(embed_dim, embed_dim)
        self.key_proj = nn.Linear(embed_dim, embed_dim)
        self.value_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

        self.kv_cache = KVCache()

    def forward(self, x, use_cache=False):
        batch_size, seq_len, embed_dim = x.size()

        # Project inputs to Q, K, V
        Q = self.query_proj(x)
        K = self.key_proj(x)
        V = self.value_proj(x)

        # Reshape for multi-head: [batch, seq_len, heads, head_dim] -> [batch, heads, seq_len, head_dim]
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        if use_cache:
            self.kv_cache.update(K, V)
            K, V = self.kv_cache.get_cache()["key"], self.kv_cache.get_cache()["value"]
            print("K", K.shape)
            print("V", V.shape)

        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)  # [batch, heads, seq_q, seq_k]

        attn_weights = F.softmax(scores, dim=-1)
        attn_output = torch.matmul(attn_weights, V)  # [batch, heads, seq_len, head_dim]

        # Concatenate heads and project out
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, embed_dim)
        output = self.out_proj(attn_output)
        return output


# Example usage
embed_dim = 32
num_heads = 4
model = SelfAttention(embed_dim, num_heads)

batch_size = 2
seq_len = 5

# Initial input (random)
x = torch.randn(batch_size, seq_len, embed_dim)

# Forward pass without cache
out = model(x, use_cache=False)
print("Output shape without cache:", out.shape)

# Simulating autoregressive generation with cache
model.kv_cache.reset()
for step in range(seq_len):
    x_step = x[:, step:step+1, :]
    #idx = torch.cat((idx, idx_next), dim=1) # (B, T + 1), normal diff between normal and cahce , 
    # in normal we go 1, 2,3, ..8 in terms of context
    # in cache we go 1, 1,  ..1 an take previous key , and v value for cache
    print(x_step.shape)
    out_step = model(x_step, use_cache=True)
    print(f"Step {step + 1}, output shape:", out_step.shape)


Output shape without cache: torch.Size([2, 5, 32])
torch.Size([2, 1, 32])
K torch.Size([2, 4, 1, 8])
V torch.Size([2, 4, 1, 8])
Step 1, output shape: torch.Size([2, 1, 32])
torch.Size([2, 1, 32])
K torch.Size([2, 4, 2, 8])
V torch.Size([2, 4, 2, 8])
Step 2, output shape: torch.Size([2, 1, 32])
torch.Size([2, 1, 32])
K torch.Size([2, 4, 3, 8])
V torch.Size([2, 4, 3, 8])
Step 3, output shape: torch.Size([2, 1, 32])
torch.Size([2, 1, 32])
K torch.Size([2, 4, 4, 8])
V torch.Size([2, 4, 4, 8])
Step 4, output shape: torch.Size([2, 1, 32])
torch.Size([2, 1, 32])
K torch.Size([2, 4, 5, 8])
V torch.Size([2, 4, 5, 8])
Step 5, output shape: torch.Size([2, 1, 32])
